In [ ]:
import pandas as pd

df = pd.read_csv("greer_archive.csv.gz", compression="gzip")
df.set_index('archive_id', inplace=True)

idx = 10000
print(df.index[idx])
print(df.iloc[idx].link)
print(df.iloc[idx].thumb)

# https://lankton.mattress.org/media/collectiveaccess/images/3/6/8/85019_ca_object_representations_media_36815_large.jpg
# https://lankton.mattress.org/viewers/apps/tilepic.php?p=https://lankton.mattress.org/media/collectiveaccess/tilepics/3/6/8/19286_ca_object_representations_media_36815_tilepic.tpc&t=5


idx = 1000
print(df.index[idx])
print(df.iloc[idx].link)
print(df.iloc[idx].thumb)

# https://lankton.mattress.org/media/collectiveaccess/images/4/1/2/99263_ca_object_representations_media_41217_large.jpg
# https://lankton.mattress.org/viewers/apps/tilepic.php?p=https://lankton.mattress.org/media/collectiveaccess/tilepics/4/1/2/74874_ca_object_representations_media_41217_tilepic.tpc&t=2


In [ ]:
from PIL import Image, ImageFile
import io
import numpy as np
import requests
from tqdm import tqdm

def getsizes(url):
    # get file size *and* image size (None if not known)
    with requests.get(url, stream=True) as response:
        response.raise_for_status() # Raise an exception for HTTP errors
    
        # 2. Initialize the incremental Pillow parser
        parser = ImageFile.Parser()
        
        # 3. Stream small chunks of the image body
        for chunk in response.iter_content(chunk_size=1024):
            if chunk:
                parser.feed(chunk)
                
                # 4. Check if the header has been successfully parsed
                if parser.image:
                    width, height = parser.image.size
                    return(width, height)
        else:
            return None

def get_pyramid(width, height, tilesize, levels):
    pyramid = []
    tiles = 0
    resolution = np.array([width, height], dtype=int)
    for i in range(levels):
        pyramid.append([resolution, np.prod((resolution - 1) // tilesize + 1)])
        resolution = np.ceil(resolution/2.0).astype(int)
    pyramid.reverse()
    return pyramid

def load_lod(url_base, pyramid, lod):
    start = 1
    resolution = pyramid[lod][0]
    grid = Image.new('RGB', size=tuple(resolution))
    for i in range(lod):
        start += pyramid[i][1]
    end = start + pyramid[lod][1]
    x, y = (0,0)
    for i in tqdm(range(start, end)):
        url = f"{url_base}{i}"
        response = requests.get(url)
        response.raise_for_status()
        tile = Image.open(io.BytesIO(response.content))
        grid.paste(tile, box=(x, y))
        x += tile.size[0]
        if x >= resolution[0]:
            y += tile.size[1]
            x = 0
    return grid

# url_base = "https://lankton.mattress.org/viewers/apps/tilepic.php?p=https://lankton.mattress.org/media/collectiveaccess/tilepics/3/6/8/19286_ca_object_representations_media_36815_tilepic.tpc&t="
# pyramid = get_pyramid(9255, 6460, 256, 8)
# img = load_lod(url_base, pyramid, 5)
# img.show()

# url_base = "https://lankton.mattress.org/viewers/apps/tilepic.php?p=https://lankton.mattress.org/media/collectiveaccess/tilepics/3/1/0/77134_ca_object_representations_media_31074_tilepic.tpc&t="
# pyramid = get_pyramid(2214, 3124, 256, 6)
# max_dim=[4096 ,4096]
# img = load_lod(url_base, pyramid, 5)
# img.show()

url_base = "https://lankton.mattress.org/viewers/apps/tilepic.php?p=https://lankton.mattress.org/media/collectiveaccess/tilepics/3/7/8/94404_ca_object_representations_media_37820_tilepic.tpc&t="
pyramid = get_pyramid(4087, 5694, 256, 7)
print(pyramid)
for i in range(14):
    print(getsizes(f"{url_base}{i}"))
img = load_lod(url_base, pyramid, 3)
img.show()

In [ ]:
from selenium import webdriver
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException

# 1. Configure Firefox Options (Optional)
options = Options()
# To run without opening a GUI window, uncomment the line below:
# options.add_argument("-headless") 

# 2. Initialize the Driver (GeckoDriver is handled automatically)
driver = webdriver.Firefox(options=options)

driver.get(f"https://lankton.mattress.org/")

In [ ]:
import time
import re
import json
import os
from selenium.common.exceptions import NoSuchElementException

def download_lods_from_url(url, driver, max_dim=[3300 ,3300]):
    wait = WebDriverWait(driver, 10)
    driver.get(url)
    button = wait.until(EC.element_to_be_clickable((By.CLASS_NAME, "zoomButton")))
    n_images = len(driver.find_elements(By.CSS_SELECTOR, "#detailRepresentationThumbnails > div"))
    n_images = n_images if n_images else 1
    images = []
    for img_idx in range(n_images):
        button = wait.until(EC.element_to_be_clickable((By.CLASS_NAME, "zoomButton")))
        button.click()
        parent = wait.until(EC.visibility_of_element_located((By.ID, "caMediaOverlayContent")))
        script = parent.find_element(By.TAG_NAME, "script").get_property("innerHTML")
        json_match = re.search(r'\.tileviewer\((.*?)\);', script)
        assert json_match
        data = json.loads(json_match.group(1))
        pyramid = get_pyramid(
            data.get("info")["width"],
            data.get("info")["height"],
            data.get("info")["tilesize"],
            data.get("info")["levels"]
        )
        lod = 0
        for i, p in enumerate(pyramid):
            lod = i
            if np.any(p[0] > max_dim):
                break
        ca_url = "https://lankton.mattress.org" + data.get("src")
        # print(ca_url, data.get("info"), lod)
        # return[]
        img = load_lod(ca_url , pyramid, lod)
        # img.thumbnail(max_dim, Image.Resampling.LANCZOS)
        images.append(img)
        if img_idx < n_images - 1:
            driver.find_element(By.CLASS_NAME, "close").click()
            wait.until(EC.invisibility_of_element_located((By.ID, "caMediaPanelContentArea")))
            driver.find_element(By.ID, "detailRepNavNext").click()
            time.sleep(0.1)
    return images

        
basedir = "images"
os.makedirs(basedir, exist_ok=True)

rng = np.random.default_rng(10)
ids = df.index[rng.integers(len(df), size=500)]
# ids = df.index[[1000]]
# ids = ["GL.S.2345"]
selection =  df.loc[ids]
images = []
start_time = time.perf_counter()
for i in range(len(selection)):
    print(selection.index[i])
    filename = selection.index[i].replace('.','-')
    pattern = re.escape(filename) + "(_[A-Z]|).jpg"
    file_exists = any(re.search(pattern, f) for f in os.listdir(basedir) if os.path.isfile(os.path.join(basedir, f)))
    if file_exists:
        print(f"Skipping {selection.index[i]}")
        continue
    images = download_lods_from_url(selection.iloc[i].link, driver)
    for j, img in enumerate(images):
        extension = f"{'_' + chr(j + 65) if len(images) > 1 else ''}.jpg" 
        img.save(os.path.join(basedir, f"{filename}{extension}"))

    # 3. Subtract the past from now
    print(f"Time elapsed: {time.perf_counter() - start_time}")

In [ ]:
import pandas as pd
import numpy as np
import re

image_ids = pd.read_csv('image_ids.txt', header=None)[0].values.tolist()
image_ids = np.unique(image_ids)
df_ids = [re.search("_media_(\d+)_medium", t)[1] for t in df['thumb'].values]
image_index = []
for image_id in image_ids:
    image_index.append(df.index[df_ids.index(str(image_id))])
# print(image_index)
print(df.iloc[0])

In [ ]:
from glob import glob

def images_to_pdf_auto_rotate(image_paths, output_pdf_path, size_inches=[8.5, 11]):
    # Standard 8.5 x 11 inches at 300 DPI (dots per inch)
    LETTER_WIDTH = int(size_inches[0] * 300)   # 2550 pixels
    LETTER_HEIGHT = int(size_inches[1] * 300)  # 3300 pixels
    
    pdf_pages = []

    for img_path in tqdm(image_paths):
        img = Image.open(img_path)
        
        if img.mode != 'RGB':
            img = img.convert('RGB')
        
        if img.width > img.height:
            img = img.rotate(270, expand=True)
        
        # 3. Create a blank white 8.5x11 canvas
        canvas = Image.new('RGB', (LETTER_WIDTH, LETTER_HEIGHT), (255, 255, 255))
        
        # 4. Scale image down proportionally to fit the canvas boundaries
        img.thumbnail((LETTER_WIDTH, LETTER_HEIGHT), Image.Resampling.LANCZOS)
        
        # 5. Center the image on the canvas
        x_offset = (LETTER_WIDTH - img.width) // 2
        y_offset = (LETTER_HEIGHT - img.height) // 2
        canvas.paste(img, (x_offset, y_offset))

        pdf_pages.append(canvas)

    if pdf_pages:
        # Save the first image and append the rest
        pdf_pages[0].save(
            output_pdf_path,
            "PDF",
            resolution=300.0,
            save_all=True,
            append_images=pdf_pages[1:]
        )
        print(f"PDF successfully saved to {output_pdf_path}")

images = glob(os.path.join('images', '*.jpg'))
images.sort()
images_to_pdf_auto_rotate(images, "images.pdf") 